In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [55]:
from openai import OpenAI
BASE_URL = "Replace with GPUstack URL" # the class model server address 
API_KEY = "Replace with API Key" # your personal API key 
client = OpenAI(base_url=BASE_URL, api_key=API_KEY) # Ask the server which model it is serving, and remember its name: 
MODEL = client.models.list().data[1].id
print("Connected. Model:", MODEL)

Connected. Model: qwen3-vl-8b-instruct


In [13]:
def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

Hello!


In [39]:
question = ( "I am writing a Python function triage_log(line) that asks an LLM to classify a " 
"security log line and returns a dataclass with label and reason. Suggest a robust " 
"prompt format and a defensive way to parse the reply. Keep it concise." ) 
print(ask_model(question))

```python
def triage_log(line):
    prompt = f"""Classify this security log line:
"{line}"
Label: [high/medium/low/unknown]
Reason: [brief justification]"""
    
    response = llm(prompt)
    parts = response.strip().split('\n')
    label = "unknown"  # default
    reason = "no reason given"
    
    for p in parts:
        if p.startswith("Label:"):
            label = p.replace("Label:", "").strip()
        elif p.startswith("Reason:"):
            reason = p.replace("Reason:", "").strip()
    
    return TriageResult(label=label, reason=reason)
```

*Robust*: handles missing fields, strips whitespace, defaults safely.
*Defensive*: no assumptions on format, minimal parsing.


In [41]:
from dataclasses import dataclass
from enum import Enum
import re
class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"
@dataclass
class Triage: 
    line: str 
    label: Label 
    reason: str
LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE) 
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)


In [42]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )
    text = ask_model(prompt)
    
    label_match = LABEL_RE.search(text) 
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"
    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN 

    reason_match = REASON_RE.search(text) 
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

In [43]:
TOOL_NAME = "LogWatch"

In [44]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='Multiple failed login attempts for the root account from an external IP suggest a brute-force attack attempt.')


In [49]:
LOG_FILE = "security_log.txt" # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20 # keep this first version small and quick
def read_log_lines(path, limit=MAX_ENTRIES): 
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()] 
    return lines[:limit]
    
entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries] 
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.txt


In [53]:
from datetime import datetime
def write_report(results, tool_name, path="assessment.md"): 
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n") 
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n") 
        f.write(f"Entries assessed: {len(results)}\n\n") 
        for r in results: 
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f" - `{r.line}`\n")
        return path 

report_path = write_report(results, TOOL_NAME) 
print("Wrote", report_path, "\n") 
print(open(report_path,encoding="utf-8").read())

Wrote assessment.md 

# LogWatch Security Log Assessment

Generated: 2026-11-23 23:03

Entries assessed: 20

- **BENIGN** The log line is incomplete and lacks specific content or context to indicate any security concern.
 - `Timestamp                       (process)[PID]`
- **BENIGN** This log records a normal system event where the laptop lid was closed, which is a standard user action and not indicative of malicious activity.
 - `2026-11-23 16:45:28.310107-0600  localhost loginwindow[422]: [com.apple.loginwindow.logging:Standard] clamshellStateChanged | Clamshell state changed: closed=1, shouldSleepWhenClosed=0`
- **BENIGN** This is a system-level notification about the clamshell state change, which is normal macOS behavior and not indicative of malicious activity.
 - `2026-11-23 16:45:28.310111-0600  localhost loginwindow[422]: [com.apple.loginwindow.logging:Standard] clamshellStateChanged | clamshellState.clamshellIsClosed == clamshellIsClosed`
- **BENIGN** This is a CoreAudio syst

 Why model the output with an Enum and dataclass instead of a bare dict?
The output is modeled with an Enum, as with dict, it allows invalid attributes to pass without crashing. Dataclass is there for readability purposes.
 What does your defensive parsing do when the model ignores the format?
Defensive parsing handles the data in such a way that prevent crashes and provides as much info in spite of the formatting.
 Which entries in your report look wrong, and how would you catch that?
An entry that looks unusual is the first one which should be skipped. In the original log, this line shows the column names. One can catch this by filtering for that incomplete output or going to the original log's line location.